## 커스텀 툴 바인딩 하기

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_agent_study"
os.environ["LANGSMITH_PROJECT"] = project_name

In [13]:
from langchain.agents import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_tavily import TavilySearch
from typing import List

In [14]:
@tool
def add_function(a: float, b: float) -> float:
    """Adds tow numbers together."""
    
    return a + b


In [15]:
@tool
def multiply_function(a: float, b: float) -> float:
    """Multiplies tow numbers together."""
    
    return a * b

In [10]:
SYSTEM_PROMPT = """
너는 도구를 사용하는 유용한 어시스턴트야. 적절한 도구를 사용해서 응답해줘.
"""

# 1. 모델 설정
llm = ChatOpenAI(
    model= "gpt-5-mini",
    temperature=0
)

# # 2. 도구 설정
tools = [add_function, multiply_function]

# 3. 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. 단일 에이전트 생성
agent = create_openai_tools_agent(
    llm=llm,
    tools = tools,
    prompt = prompt
)

# 5. excutor 설정
excutor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True
)

In [11]:
result = excutor.invoke({"input": "5 + 10은 뭐야?"})



> Entering new AgentExecutor chain...

Invoking: `add_function` with `{'a': 5, 'b': 10}`


15.05 + 10 = 15

> Finished chain.


In [14]:
result = excutor.invoke({"input": "5 * 10 은 뭐야?"})



> Entering new AgentExecutor chain...

Invoking: `multiply_function` with `{'a': 5, 'b': 10}`


50.05 * 10 은 50 이에요.

> Finished chain.


In [15]:
result = excutor.invoke({"input": "5 * 10 + 8 은 뭐야?"})



> Entering new AgentExecutor chain...

Invoking: `multiply_function` with `{'a': 5, 'b': 10}`


50.0
Invoking: `add_function` with `{'a': 10, 'b': 8}`


18.0
Invoking: `add_function` with `{'a': 50, 'b': 8}`


58.05 * 10 + 8 의 값은 58입니다.

> Finished chain.


## 커스텀 도구 추가 gpt-5-mini

In [24]:
@tool
def recommend_career(major: str, skills: List[str]) -> str:
    """Recommend a career based on user profile.
    ex) user_profile = {"major": "AI", "skills": ["Python", "React]}
    """
    if major.upper() in ["AI", "ai", "인공지능"]:
        return "원티드 사에서 채용하는 머신러닝 딥러닝 엔지니어에 지원해보세요.\n쿠팡에서 채용하는 에이전트AI 개발자에 지원해보세요."
    return "채용공고를 찾고 있습니다."

SYSTEM_PROMPT = """
너는 사용자의 커리어 방향을 도와주는 AI 어시스턴트야.

다음과 같은 상황에서는 반드시 제공된 도구를 사용해야 해:
- 사용자가 자신의 전공, 기술 스택, 또는 관심 분야를 말하며 "어떤 직무가 어울릴까?", "어떤 커리어를 추천해?", "어디 지원하면 좋을까?" 등의 질문을 할 때
- 사용자의 기술(skill), 전공(major), 프로젝트 경험 등을 기반으로 직업을 추천해야 할 때

이럴 경우에는 반드시 `recommend_career` 도구를 호출해서 추천을 생성해야 해.

도구 설명:
- `recommend_career(user_profile: dict)` : 사용자의 프로필(전공, 기술 목록 등)을 기반으로 커리어를 추천하는 함수.
  예시 입력: "major": "AI", "skills": ["Python", "React"]
  예시 출력: "원티드 사에서 채용하는 머신러닝 엔지니어에 지원해보세요."

그 외의 일반적인 질문에는 자연스럽게 대화형으로 답변해.
답변할 때는 사용자의 의도에 맞게 한국어로 부드럽고 친절하게 말해줘.
"""

In [18]:
# 1. 모델 설정
llm = ChatOpenAI(
    model= "gpt-5-mini",
    temperature=0
)

# # 2. 도구 설정
tools = [add_function, multiply_function, recommend_career]

# 3. 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("placeholder", '{chat_history}'),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 4. 단일 에이전트 생성
agent = create_openai_tools_agent(
    llm=llm,
    tools = tools,
    prompt = prompt
)

# 5. excutor 설정
excutor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True
)

In [19]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from typing import Dict
from langchain_core.chat_history import InMemoryChatMessageHistory

In [20]:
stores : Dict[str, InMemoryChatMessageHistory] = {}

def get_store(session_id: str) -> RunnableWithMessageHistory:
    if session_id not in stores:
        stores[session_id] = InMemoryChatMessageHistory()
    return stores[session_id]

agent_history = RunnableWithMessageHistory(
    excutor,
    lambda session_id: get_store(session_id),
    input_messages_key="input",
    history_messages_key="chat_history"
)


In [21]:
config = {"configurable" : {"session_id" : "user-123"}}
agent_history.invoke({
    "input": "5 + 10은 뭐야?"}, config=config)



> Entering new AgentExecutor chain...
5 + 10 = 15입니다.

> Finished chain.


{'input': '5 + 10은 뭐야?', 'chat_history': [], 'output': '5 + 10 = 15입니다.'}

In [22]:
config = {"configurable" : {"session_id" : "user-123"}}
agent_history.invoke({
    "input": "내가 방금 무슨 말 했지?"}, config=config)



> Entering new AgentExecutor chain...
방금 "5 + 10은 뭐야?"라고 물으셨고, 제가 "5 + 10 = 15입니다."라고 답했어요.

> Finished chain.


{'input': '내가 방금 무슨 말 했지?',
 'chat_history': [HumanMessage(content='5 + 10은 뭐야?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='5 + 10 = 15입니다.', additional_kwargs={}, response_metadata={})],
 'output': '방금 "5 + 10은 뭐야?"라고 물으셨고, 제가 "5 + 10 = 15입니다."라고 답했어요.'}

In [23]:
config = {"configurable" : {"session_id" : "user-123"}}
agent_history.invoke({
    "input": "ai 전공아고, python을 잘 다루는데 나에게 요즘 어떤 직무가 좋을까?"}, config=config)




> Entering new AgentExecutor chain...

Invoking: `recommend_career` with `{'major': 'AI', 'skills': ['Python']}`


원티드 사에서 채용하는 머신러닝 딥러닝 엔지니어에 지원해보세요.
쿠팡에서 채용하는 에이전트AI 개발자에 지원해보세요.원티드 사에서 채용하는 머신러닝 딥러닝 엔지니어에 지원해보세요.  
쿠팡에서 채용하는 에이전트AI 개발자에 지원해보세요.

> Finished chain.


{'input': 'ai 전공아고, python을 잘 다루는데 나에게 요즘 어떤 직무가 좋을까?',
 'chat_history': [HumanMessage(content='5 + 10은 뭐야?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='5 + 10 = 15입니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='내가 방금 무슨 말 했지?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='방금 "5 + 10은 뭐야?"라고 물으셨고, 제가 "5 + 10 = 15입니다."라고 답했어요.', additional_kwargs={}, response_metadata={})],
 'output': '원티드 사에서 채용하는 머신러닝 딥러닝 엔지니어에 지원해보세요.  \n쿠팡에서 채용하는 에이전트AI 개발자에 지원해보세요.'}